In [ ]:
import pandas as pd
import numpy as np
from cleantext import clean
import re
from transformers import XLNetTokenizer, XLNetForSequenceClassification, TrainingArguments, Trainer, pipeline
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import datasets
import evaluate
import random

In [ ]:
dataTrain = pd.read_csv('emotion-labels-train.csv')
dataTest = pd.read_csv('emotion-labels-test.csv')
dataVal = pd.read_csv('emotion-labels-val.csv')

In [ ]:
#preprocess data

In [ ]:
dataTrain.head()

In [ ]:
data = pd.concat([dataTrain, dataTest, dataVal], ignore_index=True)

In [ ]:
data['textClean'] = data['text'].apply(lambda x: clean(x, no_emoji=True))

In [ ]:
data['textClean'] = data['textClean'].apply(lambda x: re.sub('@[^\s]+', '', x))

In [ ]:
data.head(20)

In [ ]:
data['label'].value_counts().plot(kind='bar')

In [ ]:
g = data.groupby('label')
data = pd.DataFrame(g.apply(lambda x: x.sample(g.size().min()).reset_index(drop=True)))

In [ ]:
data['label'].value_counts().plot(kind='bar')

In [ ]:
data['labelInt'] = LabelEncoder().fit_transform(data['label'])

In [ ]:
numLabels = 4

In [ ]:
train_split, test_split = train_test_split(data, train_size=0.8)
train_split, val_split = train_test_split(train_split, train_size=0.9)

In [ ]:
print(len(train_split))
print(len(test_split))
print(len(val_split))

In [ ]:
train_df = pd.DataFrame({
    "label": train_split.labelInt.values,
    "text": train_split.textClean.values
})
test_df = pd.DataFrame({
    "label": test_split.labelInt.values,
    "text": test_split.textClean.values
})

In [ ]:
train_df = datasets.Dataset.from_dict(train_df)
test_df = datasets.Dataset.from_dict(test_df)

In [ ]:
dataset_dict = datasets.DatasetDict({"train": train_df, "test": test_df})

In [ ]:
dataset_dict

In [ ]:
#create Embeddings

In [ ]:
tokenizer = XLNetTokenizer.from_pretrained('xlnet-base-cased')

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples['text'], padding = 'max_length', max_length = 128, truncation = True)

In [ ]:
tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)

In [ ]:
tokenized_datasets

In [ ]:
print(tokenized_datasets['train']['text'][0])

In [ ]:
print(tokenized_datasets['train']['input_ids'][0])

In [ ]:
print(tokenized_datasets['train']['token_type_ids'][0])

In [ ]:
print(tokenized_datasets['train']['attention_mask'][0])

In [ ]:
small_train_dataset = tokenized_datasets['train'].shuffle(seed=42).select(range(100))
small_eval_dataset = tokenized_datasets['test'].shuffle(seed=42).select(range(100))

In [ ]:
model = XLNetForSequenceClassification.from_pretrained('xlnet-base-cased', num_labels = numLabels, id2label={0: 'anger', 1: 'fear', 2: 'joy', 3: 'sadness'})